> **How to use this notebook.** Step 3 of 6. Needs: 02 done (raw inputs in `DATA_DIR`, derived inputs in `DERIVED_DIR`, the work folder `WORK_DIR`). On Colab each notebook runs in its own runtime: keep `DATA_DIR` and `OUTPUTS_DIR` on Google Drive (`MOUNT_DRIVE = True`) so 03 finds what 02 wrote.
>
> **Nurnet files (included).** The site list (`nurnet_nuraghi.shp`, 7,167 nuraghi with tower counts) and the per-site building stone file are Nurnet's data, included in `data/nurnet/`; 02 copies them to the names the configuration expects (`DATA_DIR/sites/`, `DATA_DIR/stone/`) and checks their SHA-256. For the latest version and source information contact Nurnet at nurnet.presidente@gmail.com. Details: [HOW_TO_USE.md](HOW_TO_USE.md#nurnet-files).
>
> Order: 01_setup, 02_inputs, 03_scenarios, 04_analysis_report, 05_audit, 06_sensitivity (Jupyter or Google Colab). Code: the model (`code/`); configuration: `config/scenarios/config_full_audited.yaml` (the model's configuration as the queue reads it; the same values as `config/config_full_audited.yaml`). Full guide: [notebooks/HOW_TO_USE.md](HOW_TO_USE.md).

# 03 Scenarios: run any scenario, resume, and the reproduction check

Runs `model.py` (repast4py + MPI) from the repository code (the model) with the model's configuration and the
derived-input cache. Every run is defined by the
scenario name, the sites, the seeds (seed *i* = run *i*; every random draw is
`numpy.random.default_rng([seed, site id, ...])`) and nothing else: no clock, no unseeded randomness.

**Presets**

| preset | sites | seeds | scenarios | Colab time (2 ranks; estimate) |
|---|---|---|---|---|
| `demo` | the first 30 of the haul table | 1 | as `quick` | about a minute (a smoke test of the whole chain 03-04) |
| `quick` | 300 (`code/sites300_real.txt`) | 1-3 | baseline, lit webster, alliance A5_full, the headline MOST_REALISTIC_AUDITED_LIT_P10, MOST_REALISTIC_P10 (+ its `_alone` twin for 04) | minutes |
| `full` | all 7,167 | 1-100 | the same | many hours (alliance scenarios simulate the whole island on every rank; ~0.9-3 GB per rank) |
| `custom` | your `SITES` | your `SEEDS` | your `SCENARIOS` | |

Scenario names: `baseline`, `coop:NAME` (cooperation, e.g. `coop:full`), `lit:NAME` (literature chaîne
opératoire, e.g. `lit:webster`, `lit:REALISTIC`), `alliance:NAME` (adaptive alliances and MOST_REALISTIC, e.g.
`alliance:A5_full`, `alliance:MOST_REALISTIC_AUDITED_LIT_P10` (the headline); suffixes `_P10`/`_P20` phase
construction, `_sea` adds sea transport), `pre:NAME` (pre-existing networks). Section 2 lists the names in
`config.yaml`. The A5 sensitivity grid (`alliance:sensitivity`) and the Morris / Sobol designs are in 06.

**Resume.** Part files go to `RESULTS_DIR/<LABEL>/runs/` one per seed, in a folder named by the run signature
(a hash of every parameter, the site list and the code version). With `RESUME = True` a rerun skips finished
seeds, so after a disconnect you run the same cells again. Keep `RESULTS_DIR` on persistent disk for that.

**Outputs** in `RESULTS_DIR/<LABEL>/`: `baseline/`, `scenarios_lit/<name>/`, `scenarios_alliance/<name>/`, ...
each with `nuraghe_manhours_runs.csv.gz` (one row per seed and site: man-hours by stage, builders, days, years),
`nuraghe_manhours_summary.csv` (one row per site: mean/min/max/sd over seeds), and for alliance runs
`stats.json`, `strategy_by_season.csv`, `specialisation.csv`. Field meanings: `code/METHODS.md`.

**The last section can rerun the reference subset and compare it exactly with a reference you made** (see notebooks/README.md).

## Config (the only cell you edit)

All paths are set at the top of this cell. With `"auto"` the notebook uses the repository layout: `code/`,
`config/`, `data/` (raw inputs, see `data/README.md`), `data/derived/cache/` (derived inputs, see
`data/derived/README.md`) and `outputs/` (everything the notebooks write). Environment variables `NB_REPO_DIR`,
`NB_DATA_DIR`, `NB_DERIVED_DIR`, `NB_CONFIG`, `NB_OUTPUTS_DIR`, `NB_WORK_DIR`, `NB_RESULTS_DIR`, `NB_LOCK`,
`NB_INSTALL` (0 = never install) override the defaults (for headless runs with `jupyter nbconvert --execute`).
`CONFIG` ("auto") is `config/scenarios/config_full_audited.yaml`, the model's configuration: the same
values as the model's configuration `config/config_full_audited.yaml` (audited parameters, minimum haul
`transport.min_haul_flatm: 15`, near-site rule `quarry.near_site_fixed_face_m: 1000`); only `cache_dir` / `output_dir` differ.

**Google Colab:** the repository is cloned to `/content/Repast_Nuraghe` by the setup cell. Each notebook runs in its own Colab runtime, so put the data and outputs on Google Drive: `MOUNT_DRIVE = True`, `DATA_DIR = "/content/drive/MyDrive/Repast_Nuraghe/data"`, `DERIVED_DIR = "/content/drive/MyDrive/Repast_Nuraghe/data/derived/cache"`, `OUTPUTS_DIR = "/content/drive/MyDrive/Repast_Nuraghe/outputs"` (the same values in every notebook).

In [ ]:
import os

# Repository layout: the only lines you may need to change. "auto" = the default folder of the repository.
REPO_DIR = os.environ.get("NB_REPO_DIR", "auto")         # the cloned repository ("auto": the parent of notebooks/)
REPO_URL = "https://github.com/maltaweel/Repast_Nuraghe"  # Colab: cloned to /content/Repast_Nuraghe if not found
DATA_DIR = os.environ.get("NB_DATA_DIR", "auto")         # raw inputs = data_dir ("auto": REPO_DIR/data; see data/README.md)
DERIVED_DIR = os.environ.get("NB_DERIVED_DIR", "auto")   # cached derived inputs ("auto": REPO_DIR/data/derived/cache)
CONFIG = os.environ.get("NB_CONFIG", "auto")             # base config ("auto": REPO_DIR/config/scenarios/config_full_audited.yaml,
                                                         # the model's configuration)
OUTPUTS_DIR = os.environ.get("NB_OUTPUTS_DIR", "auto")   # everything written here ("auto": REPO_DIR/outputs)
WORK_DIR = os.environ.get("NB_WORK_DIR", "auto")         # fast local disk for code + cache copies ("auto": OUTPUTS_DIR/work)
RESULTS_DIR = os.environ.get("NB_RESULTS_DIR", "auto")   # where runs are kept ("auto": OUTPUTS_DIR/notebook_runs)
MOUNT_DRIVE = False                                      # Colab only: mount Google Drive at /content/drive
INSTALL = os.environ.get("NB_INSTALL", "1") == "1"      # install OpenMPI + pinned packages if this runtime lacks them
LOCK = os.environ.get("NB_LOCK") or None                 # optional flock file (shared servers only)

PRESET = os.environ.get("NB_PRESET", "quick")            # "demo" | "quick" | "full" | "custom"
SCENARIOS = ["baseline", "lit:webster", "alliance:A5_full", "alliance:MOST_REALISTIC_AUDITED_LIT_P10",
             "alliance:MOST_REALISTIC_P10", "alliance:MOST_REALISTIC_alone_P10"]
SITES = "sites300_real.txt"                              # custom: "all", a file in code/, an int N (first N), or "UID1,UID2"
SEEDS = "1-3"                                            # custom: "1-100", "1,5,9"
RANKS = int(os.environ.get("NB_RANKS", min(2, os.cpu_count() or 1)))   # MPI ranks (Colab: 2 vCPU)
RESUME = True                                            # True: skip seeds already on disk; False: rerun all (--fresh)
LABEL = None                                             # results subfolder; None = the preset name
RUN_REFERENCE_CHECK = False                              # last section: compare with OUTPUTS_DIR/reference (none shipped)
MAKE_REFERENCE = False                                   # last section: write that reference from this machine instead

## 1. Setup

In [ ]:
# Setup (no edits needed): find the repository, set the folders, install if needed, import the helpers in repro.py.
import os, sys, glob
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and MOUNT_DRIVE and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

def _is_repo(d):
    return os.path.isfile(os.path.join(d, "notebooks", "repro.py")) and os.path.isdir(os.path.join(d, "code"))

def _find_repo(p):
    cands = [p] if p != "auto" else [os.path.dirname(os.getcwd()), os.getcwd(), "/content/Repast_Nuraghe"]
    for c in cands:
        if _is_repo(c):
            return os.path.abspath(c)
    if IN_COLAB and p == "auto":                     # Colab opens only the notebook: clone the repository once
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/Repast_Nuraghe"], check=True)
        return "/content/Repast_Nuraghe"
    raise FileNotFoundError("Repository not found (tried " + ", ".join(cands) + "). Set REPO_DIR in the config cell.")

REPO_DIR = _find_repo(REPO_DIR)
PACKAGE_DIR = REPO_DIR                                     # name used by the helper calls below
sys.path.insert(0, os.path.join(REPO_DIR, "notebooks"))
import repro
if OUTPUTS_DIR == "auto":
    OUTPUTS_DIR = os.path.join(REPO_DIR, "outputs")
OUTPUTS_DIR = os.path.abspath(OUTPUTS_DIR)
L = repro.set_layout(repo=REPO_DIR, data=DATA_DIR, cache=DERIVED_DIR, config=CONFIG,
                     reference=os.path.join(OUTPUTS_DIR, "reference"))
DATA_DIR, DERIVED_DIR, CONFIG = L["data"], L["cache"], L["config"]
WORK_DIR = os.path.abspath(os.path.join(OUTPUTS_DIR, "work") if WORK_DIR == "auto" else WORK_DIR)
RESULTS_DIR = os.path.abspath(os.path.join(OUTPUTS_DIR, "notebook_runs") if RESULTS_DIR == "auto" else RESULTS_DIR)
for k in ("REPO_DIR", "DATA_DIR", "DERIVED_DIR", "CONFIG", "WORK_DIR", "RESULTS_DIR"):
    print(f"{k:12s}", globals()[k])
status = repro.ensure_env(install=INSTALL)
print("MPI:", repro.mpi_report(), "| Python", sys.version.split()[0], "| CPUs", os.cpu_count())
assert status == "ok", "Restart the kernel now (see message above), then run this notebook again."

## 2. Available scenarios

In [ ]:
# presets -> sites, seeds, label; the work folder of 02 (made here if this runtime has none)
PRESETS = {"demo": (30, "1"), "quick": ("sites300_real.txt", "1-3"), "full": ("all", "1-100")}
if PRESET in PRESETS:
    SITES, SEEDS = PRESETS[PRESET]
elif PRESET != "custom":
    raise ValueError(f"PRESET must be demo, quick, full or custom, not {PRESET!r}")
LABEL = LABEL or PRESET
if not os.path.isfile(os.path.join(WORK_DIR, "config.yaml")):
    print("no WORK_DIR/config.yaml (02_inputs not run in this runtime): making the work folder now")
    repro.prepare_inputs()      # only if inputs are missing (no DERIVED_DIR on Drive): as 02, incl. the ~30-45 min rebuild
    repro.make_workdir(PACKAGE_DIR, WORK_DIR, cache="prebuilt")
print(f"preset {PRESET}: sites {SITES}, seeds {SEEDS}, label {LABEL}, scenarios {SCENARIOS}")
for fam, names in repro.list_scenarios(WORK_DIR).items():
    print(f"{fam:9s}", names if isinstance(names, str) else ", ".join(names))

## 3. Run
Each scenario prints the model's own log. The first run compiles the numba kernels (once per runtime). Alliance runs on the full island: keep `RANKS` x ~1-3 GB under the runtime's memory.

In [ ]:
import time
t0 = time.time()
for spec in SCENARIOS:
    print("=" * 30, spec)
    out = repro.run_scenario(WORK_DIR, RESULTS_DIR, LABEL, spec, SITES, SEEDS, ranks=RANKS, resume=RESUME, lock=LOCK)
print(f"all scenarios done in {time.time() - t0:.0f} s")

## 4. Island totals of these runs

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
tab = repro.island_table(os.path.join(RESULTS_DIR, LABEL))
tab

## 5. Reproduction check (optional)
Reruns the reference subset **from scratch** (`--fresh`, own folder `RESULTS_DIR/_refcheck`): 300 sites
(`code/sites300_real.txt`), seeds 1-3, scenarios `baseline`, `lit:webster`, `alliance:A5_full`,
`alliance:MOST_REALISTIC_AUDITED_LIT_P10`, with `RANKS` ranks, and compares every output with a fingerprint file
`OUTPUTS_DIR/reference/reference_sha256.csv` (merged outputs byte for byte, gzip files by decompressed bytes,
model.py part files as the sorted set of full-precision rows; signature folder names are part of the key).

**No reference is shipped in the repository** (a fingerprint applies only to the code version, configuration and
derived inputs it was made with: here `code/` and `config/scenarios/config_full_audited.yaml`). To make one: run this section once with
`MAKE_REFERENCE = True` on a trusted machine (it writes the fingerprint with `repro.write_fingerprint`), then
compare later runs against it with `RUN_REFERENCE_CHECK = True`. A check passes only if every item matches and
nothing is missing or extra; the rank count does not change the items.

In [ ]:
if MAKE_REFERENCE or RUN_REFERENCE_CHECK:
    chk = os.path.join(RESULTS_DIR, "_refcheck")
    t0 = time.time()
    repro.run_reference(WORK_DIR, chk, ranks=RANKS, lock=LOCK, fresh=True)
    ref_csv = os.path.join(repro.LAYOUT["reference"], "reference_sha256.csv")
    if MAKE_REFERENCE:
        os.makedirs(os.path.dirname(ref_csv), exist_ok=True)
        repro.write_fingerprint(repro.fingerprint(chk), ref_csv)
        print("reference written:", ref_csv)
    else:
        assert os.path.isfile(ref_csv), f"no reference at {ref_csv}: make one first (MAKE_REFERENCE = True)"
        ok = repro.compare_reference(chk, ref_csv)
        print(f"rerun + compare: {time.time() - t0:.0f} s")
        print("REPRODUCTION CHECK:", "PASSED - exact match with the reference" if ok else "FAILED")
        assert ok, "REPRODUCTION CHECK FAILED: outputs differ from the reference (see the list above)"
else:
    print("RUN_REFERENCE_CHECK and MAKE_REFERENCE are False: skipped")

Runs done. Next: `04_analysis_report.ipynb`, then `05_audit.ipynb`.